# Gene Set Enrichment Analysis with RNAMultiOmics

This notebook demonstrates the comprehensive gene set enrichment analysis capabilities of the RNAMultiOmics package, including:

1. **Over-Representation Analysis (ORA)** - Fisher's exact test and hypergeometric test
2. **Gene Set Enrichment Analysis (GSEA)** - Ranked gene list analysis
3. **Gene Ontology (GO) Analysis** - Using gseapy and goatools
4. **PAGE (Pathway Analysis of Gene Expression)** - Mutual information-based analysis
5. **Utility Functions** - Gene list processing and validation

The module is designed to work progressively based on available dependencies:
- **Basic mode**: Works with standard Python libraries (pandas, numpy, scipy)
- **Enhanced mode**: Additional functionality with gseapy
- **Advanced mode**: Full features with goatools and bio-pypage

## Setup and Dependencies

First, let's check what dependencies are available and import the necessary modules.

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path for importing multiomics
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Check available dependencies
print("Checking dependencies...")
dependencies = {}

try:
    import scipy
    dependencies['scipy'] = True
    print("✓ scipy is available")
except ImportError:
    dependencies['scipy'] = False
    print("✗ scipy is not available")

try:
    import gseapy
    dependencies['gseapy'] = True
    print("✓ gseapy is available")
except ImportError:
    dependencies['gseapy'] = False
    print("✗ gseapy is not available (install with: pip install gseapy)")

try:
    from goatools.obo_parser import GODag
    dependencies['goatools'] = True
    print("✓ goatools is available")
except ImportError:
    dependencies['goatools'] = False
    print("✗ goatools is not available (install with: pip install goatools)")

try:
    import statsmodels
    dependencies['statsmodels'] = True
    print("✓ statsmodels is available")
except ImportError:
    dependencies['statsmodels'] = False
    print("✗ statsmodels is not available (install with: pip install statsmodels)")

# Check pypage availability through the wrapper
try:
    from multiomics.enrichment import check_pypage_availability
    dependencies['pypage'] = check_pypage_availability()
    if dependencies['pypage']:
        print("✓ bio-pypage is available")
    else:
        print("✗ bio-pypage is not available (install with: pip install bio-pypage)")
except ImportError:
    dependencies['pypage'] = False
    print("✗ bio-pypage import failed")

print(f"\nAvailable dependencies: {sum(dependencies.values())}/{len(dependencies)}")

## Create Sample Data

Let's create realistic sample gene expression data that mimics what you might get from RNA-seq analysis.

In [ ]:
def create_sample_expression_data():
    """Create sample gene expression data for demonstration"""
    print("Creating sample gene expression data...")
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Use biologically relevant gene names
    genes = [
        # Housekeeping genes
        'GAPDH', 'ACTB', 'TUBB', 'HPRT1', 'GUSB',
        # Cancer-related genes
        'TP53', 'MYC', 'EGFR', 'KRAS', 'PIK3CA', 'AKT1', 'MTOR', 'PTEN',
        # DNA repair genes
        'BRCA1', 'BRCA2', 'ATM', 'CHEK2', 'PALB2', 'MLH1', 'MSH2', 'MSH6', 'PMS2',
        # Cell cycle genes
        'CDKN2A', 'RB1', 'CCND1', 'CDK4', 'CDK6', 'E2F1', 'E2F3', 'PCNA', 'CCNE1',
        # TGF-beta pathway
        'TGFB1', 'SMAD2', 'SMAD3', 'SMAD4', 'TGFBR1', 'TGFBR2',
        # Wnt pathway
        'WNT1', 'WNT3A', 'CTNNB1', 'APC', 'AXIN1', 'AXIN2', 'GSK3B',
        # Additional genes
        'NOTCH1', 'NOTCH2', 'JAK2', 'STAT3', 'NF1', 'NF2', 'VHL', 'IDH1', 'IDH2'
    ]
    
    # Create sample names
    control_samples = [f'Control_{i}' for i in range(1, 6)]  # 5 control samples
    treatment_samples = [f'Treatment_{i}' for i in range(1, 6)]  # 5 treatment samples
    all_samples = control_samples + treatment_samples
    
    # Create expression data (log2 TPM values)
    expression_data = pd.DataFrame(
        np.random.normal(5, 2, (len(genes), len(all_samples))),
        index=genes,
        columns=all_samples
    )
    
    # Add differential expression patterns
    # Upregulate oncogenes in treatment
    oncogenes = ['MYC', 'EGFR', 'KRAS', 'PIK3CA', 'AKT1', 'CCND1', 'CDK4']
    for gene in oncogenes:
        if gene in expression_data.index:
            expression_data.loc[gene, treatment_samples] += np.random.normal(2, 0.5, len(treatment_samples))
    
    # Downregulate tumor suppressors in treatment
    tumor_suppressors = ['TP53', 'BRCA1', 'BRCA2', 'PTEN', 'RB1', 'CDKN2A', 'APC']
    for gene in tumor_suppressors:
        if gene in expression_data.index:
            expression_data.loc[gene, treatment_samples] -= np.random.normal(1.5, 0.3, len(treatment_samples))
    
    print(f"Created expression data: {expression_data.shape[0]} genes × {expression_data.shape[1]} samples")
    
    return expression_data, control_samples, treatment_samples

def create_sample_gene_sets():
    """Create biologically relevant gene sets for demonstration"""
    print("Creating sample gene sets...")
    
    gene_sets = {
        'p53_Pathway': ['TP53', 'ATM', 'CHEK2', 'CDKN2A', 'RB1'],
        'PI3K_AKT_Pathway': ['PIK3CA', 'AKT1', 'MTOR', 'PTEN'],
        'Cell_Cycle': ['CCND1', 'CDK4', 'CDK6', 'E2F1', 'E2F3', 'PCNA', 'CCNE1', 'RB1'],
        'DNA_Repair': ['BRCA1', 'BRCA2', 'ATM', 'CHEK2', 'PALB2', 'MLH1', 'MSH2', 'MSH6'],
        'TGF_Beta_Pathway': ['TGFB1', 'SMAD2', 'SMAD3', 'SMAD4', 'TGFBR1', 'TGFBR2'],
        'Wnt_Pathway': ['WNT1', 'WNT3A', 'CTNNB1', 'APC', 'AXIN1', 'AXIN2', 'GSK3B'],
        'Oncogenes': ['MYC', 'EGFR', 'KRAS', 'PIK3CA', 'AKT1', 'CCND1'],
        'Tumor_Suppressors': ['TP53', 'BRCA1', 'BRCA2', 'PTEN', 'RB1', 'CDKN2A', 'APC'],
        'Housekeeping': ['GAPDH', 'ACTB', 'TUBB', 'HPRT1', 'GUSB']
    }
    
    print(f"Created {len(gene_sets)} sample gene sets")
    return gene_sets

# Create the sample data
expression_data, control_samples, treatment_samples = create_sample_expression_data()
gene_sets = create_sample_gene_sets()

# Display basic information
print("\nExpression data preview:")
print(expression_data.head())
print("\nGene sets:")
for name, genes in gene_sets.items():
    print(f"  {name}: {len(genes)} genes")

## 1. Utility Functions

Let's start by demonstrating the utility functions for gene list processing and validation.

In [ ]:
from multiomics.enrichment._utils import prepare_gene_list, clean_gene_name, filter_gene_sets

print("UTILITY FUNCTIONS DEMONSTRATION")
print("=" * 50)

# Test gene list preparation
print("\n1. Gene List Preparation:")
messy_gene_list = ['TP53', 'tp53', ' BRCA1 ', 'BRCA1.1', 'brca2', 'MYC', 'myc', None, '']
cleaned_genes = prepare_gene_list(messy_gene_list, remove_duplicates=True, verbose=True)
print(f"Original: {messy_gene_list}")
print(f"Cleaned:  {cleaned_genes}")

# Test gene name cleaning
print("\n2. Gene Name Cleaning:")
test_names = [' tp53 ', 'BRCA1.2', 'Gene_ABC', 'xyz789.version1']
for name in test_names:
    cleaned = clean_gene_name(name)
    print(f"  '{name}' -> '{cleaned}'")

# Test gene set filtering
print("\n3. Gene Set Filtering:")
all_genes = list(expression_data.index)
filtered_sets = filter_gene_sets(
    gene_sets=gene_sets,
    gene_list=all_genes,
    min_overlap=2,
    min_size=3,
    max_size=20,
    verbose=True
)

print("\nFiltered gene sets:")
for name, genes in filtered_sets.items():
    overlap = len(set(genes) & set(all_genes))
    print(f"  {name}: {len(genes)} genes, {overlap} overlap with expression data")

## 2. Over-Representation Analysis (ORA)

ORA tests for statistically significant overlap between a gene list of interest and predefined gene sets.

In [ ]:
from multiomics.enrichment.ora import run_custom_ora, format_ora_results

print("OVER-REPRESENTATION ANALYSIS (ORA)")
print("=" * 50)

# Create a list of "differentially expressed" genes
# In practice, this would come from your differential expression analysis
from multiomics.enrichment.gsea import create_ranked_list

# Create ranked gene list first
ranked_genes = create_ranked_list(
    data=expression_data,
    group1_samples=treatment_samples,
    group2_samples=control_samples,
    method='log2fc',
    verbose=False
)

# Get top upregulated genes (top 20%)
upregulated_threshold = ranked_genes.quantile(0.8)
upregulated_genes = ranked_genes[ranked_genes > upregulated_threshold].index.tolist()

# Get top downregulated genes (bottom 20%)
downregulated_threshold = ranked_genes.quantile(0.2)
downregulated_genes = ranked_genes[ranked_genes < downregulated_threshold].index.tolist()

print(f"Upregulated genes ({len(upregulated_genes)}): {upregulated_genes}")
print(f"Downregulated genes ({len(downregulated_genes)}): {downregulated_genes}")

# Run ORA for upregulated genes
print("\nRunning ORA for upregulated genes...")
ora_results_up = run_custom_ora(
    study_genes=upregulated_genes,
    gene_sets=gene_sets,
    background_genes=list(expression_data.index),
    method='fisher',
    alpha=0.05,
    verbose=True
)

print("\nORA Results for Upregulated Genes:")
print(ora_results_up[['Term', 'Overlap', 'Gene_Set_Size', 'P_value', 'Adjusted_P_value', 'Enrichment_Score', 'Significant']])

# Run ORA for downregulated genes
print("\nRunning ORA for downregulated genes...")
ora_results_down = run_custom_ora(
    study_genes=downregulated_genes,
    gene_sets=gene_sets,
    background_genes=list(expression_data.index),
    method='fisher',
    alpha=0.05,
    verbose=False
)

print("\nORA Results for Downregulated Genes:")
print(ora_results_down[['Term', 'Overlap', 'Gene_Set_Size', 'P_value', 'Adjusted_P_value', 'Enrichment_Score', 'Significant']])

# Format and display significant results
significant_up = format_ora_results(ora_results_up, significance_cutoff=0.1)
significant_down = format_ora_results(ora_results_down, significance_cutoff=0.1)

print(f"\nSignificant upregulated pathways (FDR < 0.1): {len(significant_up)}")
if not significant_up.empty:
    print(significant_up[['Term', 'Overlap', 'P_value', 'Adjusted_P_value']])

print(f"\nSignificant downregulated pathways (FDR < 0.1): {len(significant_down)}")
if not significant_down.empty:
    print(significant_down[['Term', 'Overlap', 'P_value', 'Adjusted_P_value']])

## 3. Ranked Gene Analysis and GSEA

GSEA analyzes whether genes in a pathway are enriched at the top or bottom of a ranked gene list.

In [ ]:
from multiomics.enrichment.gsea import create_ranked_list, run_preranked_gsea

print("GENE SET ENRICHMENT ANALYSIS (GSEA)")
print("=" * 50)

# Create ranked gene list
print("Creating ranked gene list (Treatment vs Control)...")
ranked_genes = create_ranked_list(
    data=expression_data,
    group1_samples=treatment_samples,  # Numerator
    group2_samples=control_samples,    # Denominator
    method='log2fc',
    verbose=True
)

print(f"\nTop 10 upregulated genes (Treatment vs Control):")
top_up = ranked_genes.head(10)
for gene, score in top_up.items():
    print(f"  {gene}: {score:.3f}")

print(f"\nTop 10 downregulated genes (Treatment vs Control):")
top_down = ranked_genes.tail(10)
for gene, score in top_down.items():
    print(f"  {gene}: {score:.3f}")

# Run preranked GSEA if gseapy is available
if dependencies['gseapy']:
    print("\nRunning preranked GSEA...")
    
    try:
        gsea_results = run_preranked_gsea(
            ranked_genes=ranked_genes,
            gene_sets=gene_sets,
            permutation_num=100,  # Reduced for faster execution
            verbose=True
        )
        
        print("\nGSEA Results:")
        print(gsea_results[['Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'FWER p-val']].head(10))
        
        # Show significant results
        significant_gsea = gsea_results[gsea_results['FDR q-val'] < 0.25]  # Standard GSEA cutoff
        print(f"\nSignificant pathways (FDR < 0.25): {len(significant_gsea)}")
        if not significant_gsea.empty:
            print(significant_gsea[['Term', 'ES', 'NES', 'FDR q-val']])
            
    except Exception as e:
        print(f"GSEA analysis failed: {e}")
        print("This might be due to insufficient gene overlap or other data issues.")
else:
    print("\nGSEA requires gseapy. Install with: pip install gseapy")
    print("However, you can still use the ranked gene list for manual analysis.")

## 4. Gene Ontology (GO) Analysis

GO analysis provides biological interpretation using the Gene Ontology database.

In [ ]:
from multiomics.enrichment.go_analysis import run_go_enrichment

print("GENE ONTOLOGY (GO) ANALYSIS")
print("=" * 50)

# Prepare gene list for GO analysis
test_genes = upregulated_genes if upregulated_genes else list(expression_data.index)[:10]
print(f"Running GO analysis on {len(test_genes)} genes: {test_genes}")

if dependencies['gseapy']:
    print("\nRunning GO enrichment using gseapy...")
    
    try:
        go_results = run_go_enrichment(
            gene_list=test_genes,
            organism='human',
            method='gseapy',
            categories=['GO_Biological_Process_2021'],  # Focused on one category for demo
            background=list(expression_data.index),
            verbose=True
        )
        
        if go_results is not None and not go_results.empty:
            print("\nGO Enrichment Results (top 10):")
            print(go_results[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head(10))
            
            # Show significant results
            significant_go = go_results[go_results['Adjusted P-value'] < 0.05]
            print(f"\nSignificant GO terms (FDR < 0.05): {len(significant_go)}")
            if not significant_go.empty:
                print(significant_go[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head())
        else:
            print("No GO enrichment results returned. This might be due to:")
            print("- Limited gene overlap with GO databases")
            print("- Network connectivity issues")
            print("- Gene name format incompatibility")
            
    except Exception as e:
        print(f"GO analysis failed: {e}")
        print("This is common when running offline or with limited internet access.")
        
elif dependencies['goatools']:
    print("\ngoatools is available but requires additional setup.")
    print("See the enrichment module documentation for goatools configuration.")
    
else:
    print("\nGO analysis requires gseapy or goatools.")
    print("Install with: pip install gseapy  # or pip install goatools")
    print("\nNote: GO analysis often requires internet connectivity for database access.")

## 5. PAGE (Pathway Analysis of Gene Expression)

PAGE uses mutual information to identify pathways that are informative about expression profiles.

In [ ]:
if dependencies['pypage']:
    from multiomics.enrichment import run_page, get_pypage_info
    
    print("PAGE (PATHWAY ANALYSIS OF GENE EXPRESSION)")
    print("=" * 50)
    
    # Get pypage info
    info = get_pypage_info()
    print(f"pypage info: {info}")
    
    print("\nPAGE uses mutual information to identify informative pathways.")
    print("Unlike traditional enrichment methods, PAGE considers the full expression")
    print("distribution and identifies pathways that help distinguish sample groups.")
    
    # Prepare data for PAGE analysis
    print(f"\nRunning PAGE analysis on {expression_data.shape[0]} genes and {expression_data.shape[1]} samples...")
    
    try:
        page_results = run_page(
            expression_data=expression_data,
            gene_sets=gene_sets,
            n_shuffle=500,  # Number of permutations for significance testing
            alpha=0.05,     # Significance threshold
            n_bins=10,      # Number of expression bins
            verbose=True
        )
        
        print("\nPAGE Results:")
        print(page_results[['Term', 'MI', 'P_value', 'Informative', 'N_shared']]) 
        
        # Show informative pathways
        informative_pathways = page_results[page_results['Informative'] == True]
        print(f"\nInformative pathways (α < 0.05): {len(informative_pathways)}")
        
        if not informative_pathways.empty:
            print("\nTop informative pathways:")
            print(informative_pathways[['Term', 'MI', 'P_value']].sort_values('MI', ascending=False))
            
            print("\nInterpretation:")
            print("- MI (Mutual Information): Higher values indicate more informative pathways")
            print("- Informative pathways help distinguish between sample conditions")
            print("- This complements traditional enrichment by considering expression patterns")
        else:
            print("No pathways reached significance threshold.")
            print("This might indicate:")
            print("- Limited differential expression between conditions")
            print("- Small sample size")
            print("- Need for different expression binning parameters")
            
    except Exception as e:
        print(f"PAGE analysis failed: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("PAGE (PATHWAY ANALYSIS OF GENE EXPRESSION)")
    print("=" * 50)
    print("\nPAGE analysis requires bio-pypage.")
    print("Install with: pip install bio-pypage")
    print("\nPAGE uses mutual information to identify pathways that are")
    print("informative about expression profiles, providing a different")
    print("perspective from traditional enrichment methods.")

## 6. Integration Example: Multi-Method Analysis

Let's combine multiple methods to get a comprehensive view of pathway activity.

In [ ]:
print("INTEGRATED MULTI-METHOD ANALYSIS")
print("=" * 50)

# Compile results from different methods
analysis_summary = []

# Add ORA results
for _, row in ora_results_up.iterrows():
    analysis_summary.append({
        'Pathway': row['Term'],
        'Method': 'ORA_Upregulated',
        'Score': row['Enrichment_Score'],
        'P_value': row['P_value'],
        'Adjusted_P_value': row['Adjusted_P_value'],
        'Significant': row['Significant']
    })

for _, row in ora_results_down.iterrows():
    analysis_summary.append({
        'Pathway': row['Term'],
        'Method': 'ORA_Downregulated', 
        'Score': row['Enrichment_Score'],
        'P_value': row['P_value'],
        'Adjusted_P_value': row['Adjusted_P_value'],
        'Significant': row['Significant']
    })

# Add GSEA results if available
if dependencies['gseapy'] and 'gsea_results' in locals():
    for _, row in gsea_results.iterrows():
        analysis_summary.append({
            'Pathway': row['Term'],
            'Method': 'GSEA',
            'Score': row['NES'],
            'P_value': row['NOM p-val'],
            'Adjusted_P_value': row['FDR q-val'],
            'Significant': row['FDR q-val'] < 0.25
        })

# Add PAGE results if available
if dependencies['pypage'] and 'page_results' in locals():
    for _, row in page_results.iterrows():
        analysis_summary.append({
            'Pathway': row['Term'],
            'Method': 'PAGE',
            'Score': row['MI'],
            'P_value': row['P_value'],
            'Adjusted_P_value': row['P_value'],  # PAGE doesn't do multiple testing correction by default
            'Significant': row['Informative']
        })

# Create summary DataFrame
summary_df = pd.DataFrame(analysis_summary)

if not summary_df.empty:
    print("\nComprehensive Analysis Summary:")
    print(summary_df.pivot_table(
        index='Pathway', 
        columns='Method', 
        values='Significant', 
        fill_value=False
    ))
    
    # Find pathways significant in multiple methods
    pathway_counts = summary_df[summary_df['Significant']].groupby('Pathway').size()
    multi_significant = pathway_counts[pathway_counts > 1]
    
    print(f"\nPathways significant in multiple methods:")
    if not multi_significant.empty:
        for pathway, count in multi_significant.items():
            methods = summary_df[(summary_df['Pathway'] == pathway) & 
                               (summary_df['Significant'])]['Method'].tolist()
            print(f"  {pathway}: {methods}")
    else:
        print("  None found (this is common with small sample sizes)")
        
    print("\nMethod-specific insights:")
    print("- ORA: Tests for significant overlap with gene lists")
    print("- GSEA: Considers ranking and correlation structure")
    print("- PAGE: Identifies pathways informative about expression patterns")
    print("- GO: Provides standardized biological interpretation")
    
else:
    print("No analysis results available for summary.")
    print("This might be due to missing dependencies or data issues.")

## Summary and Next Steps

This notebook demonstrated the comprehensive enrichment analysis capabilities of RNAMultiOmics.

In [ ]:
print("ENRICHMENT ANALYSIS SUMMARY")
print("=" * 50)

print("✓ Methods demonstrated:")
print("  - Utility functions for gene list processing")
print("  - Over-representation analysis (ORA) with Fisher's exact test")
print("  - Ranked gene list creation for GSEA")
if dependencies['gseapy']:
    print("  - Gene Set Enrichment Analysis (GSEA) with gseapy")
    print("  - Gene Ontology (GO) enrichment analysis")
if dependencies['pypage']:
    print("  - PAGE (Pathway Analysis of Gene Expression) with mutual information")

print("\n✓ Key features:")
print("  - Works with different dependency configurations")
print("  - Handles multiple statistical methods")
print("  - Provides consistent output formats")
print("  - Includes proper multiple testing correction")
print("  - Supports custom gene sets and databases")

print("\n✓ Integration points:")
print("  - Input: Gene expression data from RNA-seq or microarray")
print("  - Gene sets: Custom pathways, MSigDB, GO, KEGG, etc.")
print("  - Output: Standardized enrichment results for downstream analysis")

print("\n📋 To use in your own analysis:")
print("1. Install dependencies based on your needs:")
print("   - Basic: pip install scipy statsmodels")
print("   - Enhanced: pip install gseapy")
print("   - Advanced: pip install goatools bio-pypage")
print("2. Prepare your gene expression data as a pandas DataFrame")
print("3. Define your gene sets of interest")
print("4. Run the appropriate enrichment analysis methods")
print("5. Interpret results in biological context")

print("\n🔗 Additional resources:")
print("  - Module documentation: src/multiomics/enrichment/README.md")
  print("  - Test suite: tests/ directory")
print("  - MSigDB gene sets: https://www.gsea-msigdb.org/gsea/msigdb/")
print("  - Gene Ontology: http://geneontology.org/")

print(f"\nDependencies status: {sum(dependencies.values())}/{len(dependencies)} available")
for dep, available in dependencies.items():
    status = "✓" if available else "✗"
    print(f"  {status} {dep}")